# Kujenga na Mifano ya Mistral 

## Utangulizi 

Somo hili litajumuisha: 
- Kuchunguza Mifano tofauti ya Mistral 
- Kuelewa matumizi na hali za kila mfano 
- Sampuli za nambari zinaonyesha sifa za kipekee za kila mfano. 


## Modelle za Mistral 

Katika somo hili, tutaangazia Modelle 3 tofauti za Mistral: 
**Mistral Large**, **Mistral Small** na **Mistral Nemo**. 

Kila moja ya modeli hizi inapatikana bure kwenye [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst). Msimbo katika daftari hili utatumia modeli hizi kuendesha msimbo.

> **Kumbuka:** GitHub Models itasimama mwishoni mwa Julai 2026. Hapa kuna maelezo zaidi kuhusu kutumia [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) kufanikisha majaribio na modeli za AI.


## Mistral Large 2 (2407)
Mistral Large 2 kwa sasa ni mfano wa bendera kutoka Mistral na umebuniwa kwa matumizi ya kampuni. 

Mfano huu ni toleo lililoboreshwa la Mistral Large asilia kwa kutoa 
-  Dirisha kubwa la muktadha - 128k dhidi ya 32k 
-  Utendaji bora katika Kazi za Hisabati na Uandishi wa Msimbo - usahihi wa wastani wa 76.9% dhidi ya 60.4% 
-  Kuongezeka kwa utendaji wa lugha nyingi - lugha ni pamoja na: Kiingereza, Kifaransa, Kijerumani, Kihispania, Kitaliano, Kireno, Kiholanzi, Kirusi, Kichina, Kijapani, Kikorea, Kiarabu, na Kihindi.

Kwa vipengele hivi, Mistral Large hutawala katika 
- *Uundaji uliosaidiwa na Urejeshaji (RAG)* - kutokana na dirisha kubwa la muktadha
- *Kupiga Simu za Kazi* - mfano huu una uwezo wa asili wa kupiga simu za kazi ambazo huwezesha kuunganishwa na zana za nje na APIs. Simu hizi zinaweza kufanyika sambamba au moja baada ya nyingine kwa mpangilio wa mfululizo. 
- *Uundaji wa Msimbo* - mfano huu hutawala katika uundaji wa Python, Java, TypeScript na C++. 


### Mfano wa RAG ukitumia Mistral Large 2 


Katika mfano huu, tunatumia Mistral Large 2 kuendesha muundo wa RAG juu ya hati ya maandishi. Swali limeandikwa kwa Kikorea na linauliza kuhusu shughuli za mwandishi kabla ya chuo kikuu.

Inatumia Mfano wa Cohere Embeddings kuunda embeddings za hati ya maandishi pamoja na swali. Kwa sampuli hii, inatumia kifurushi cha Python cha faiss kama hifadhi ya vekta.

Ombi linalotumwa kwa mfano wa Mistral linajumuisha maswali pamoja na vipande vilivyopatikana vinavyofanana na swali. Kisha Mfano hutoa jibu la lugha ya asili.


In [50]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
import numpy as np
import faiss
import os

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.ai.inference import EmbeddingsClient

# Get these from your Microsoft Foundry project's "Overview" page
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Mistral-large"
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt')
text = response.text

chunk_size = 2048
chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]
len(chunks)

embed_model_name = "cohere-embed-v3-multilingual" 

embed_client = EmbeddingsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(token)
)

embed_response = embed_client.embed(
    input=chunks,
    model=embed_model_name
)



text_embeddings = []
for item in embed_response.data:
    length = len(item.embedding)
    text_embeddings.append(item.embedding)
text_embeddings = np.array(text_embeddings)


d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)

question = "저자가 대학에 오기 전에 주로 했던 두 가지 일은 무엇이었나요?？"

question_embedding = embed_client.embed(
    input=[question],
    model=embed_model_name
)

question_embeddings = np.array(question_embedding.data[0].embedding)


D, I = index.search(question_embeddings.reshape(1, -1), k=2) # distance, index
retrieved_chunks = [chunks[i] for i in I.tolist()[0]]

prompt = f"""
Context information is below.
---------------------
{retrieved_chunks}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {question}
Answer:
"""


chat_response = client.complete(
    messages=[
        SystemMessage(content="You are a helpful assistant."),
        UserMessage(content=prompt),
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model=model_name
)

print(chat_response.choices[0].message.content)


The author primarily engaged in two activities before college: writing and programming. In terms of writing, they wrote short stories, albeit not very good ones, with minimal plot and characters expressing strong feelings. For programming, they started writing programs on the IBM 1401 used for data processing during their 9th grade, at the age of 13 or 14. They used an early version of Fortran and typed programs on punch cards, later loading them into the card reader to run the program.


## Mistral Ndogo 
Mistral Ndogo ni mfano mwingine katika familia ya mifano ya Mistral chini ya kundi la premier/enterprise. Kama jina linavyoonyesha, mfano huu ni Mfano Mdogo wa Lugha (SLM). Faida za kutumia Mistral Ndogo ni kwamba ni:
- Kuokoa Gharama ikilinganishwa na Mistral LLMs kama Mistral Large na NeMo - punguzaji la bei la 80%
- Latency ya chini - majibu ya haraka ukilinganisha na LLMs za Mistral
- Inayobadilika - inaweza kuanzishwa katika mazingira tofauti bila vizuizi vikubwa kwenye rasilimali zinazohitajika.


Mistral Ndogo ni nzuri kwa:
- Kazi za msingi za maandishi kama muhtasari, uchambuzi wa hisia na tafsiri.
- Programu ambapo maombi ya mara kwa mara hutolewa kutokana na ufanisi wake wa gharama
- Kazi za msimbo zenye latency ya chini kama ukaguzi na mapendekezo ya msimbo


## Kulinganisha Mistral Small na Mistral Large 

Kuonyesha tofauti katika ucheleweshaji kati ya Mistral Small na Large, endesha seli zilizo hapa chini. 

Unapaswa kuona tofauti katika nyakati za majibu kati ya sekunde 3-5. Pia angalia urefu wa majibu na mtindo kwa maelezo sawa.  


In [ ]:
import os 
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Mistral-small"
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(content="You are a helpful coding assistant."),
        UserMessage(content="Can you write a Python function to the fizz buzz test?"),
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model=model_name
)

print(response.choices[0].message.content)


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Mistral-large"
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(content="You are a helpful coding assistant."),
        UserMessage(content="Can you write a Python function to the fizz buzz test?"),
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model=model_name
)

print(response.choices[0].message.content)


## Mistral NeMo

Ukilinganisha na modeli nyingine mbili zilizojadiliwa katika somo hili, Mistral NeMo ni modeli pekee ya bure yenye Leseni ya Apache2.

Inachukuliwa kama sasisho la LLM ya awali ya chanzo huria kutoka Mistral, Mistral 7B.

Baadhi ya sifa nyingine za modeli ya NeMo ni:

- *Utokanishaji Bora Zaidi:* Modeli hii inatumia tokenizer ya Tekken badala ya tiktoken inayotumika mara nyingi. Hii inaruhusu utendaji bora kwa lugha na msimbo zaidi.

- *Uboreshaji wa Kuzaidi (Finetuning):* Modeli ya msingi inapatikana kwa uboreshaji wa zaidi. Hii inaruhusu ufanisi zaidi kwa matumizi ambapo uboreshaji unaweza kuhitajika.

- *Kupiga Simu za Kazi Asilia* - Kama Mistral Large, modeli hii imefunzwa kwenye kupiga simu za kazi. Hii inafanya iwe ya kipekee kwa kuwa mojawapo ya modeli za mwanzo za chanzo huria kufanya hivyo.


## Mistral NeMo

Compared to the other two models discussed in this lesson, Mistral NeMo is the only free model with an Apache2 License. 

It is viewed as an upgrade to the earlier open source LLM from Mistral, Mistral 7B. 

Some other feature of the NeMo model are: 

- *More efficient tokenization:* This model uses the Tekken tokenizer over the more commonly used tiktoken. This allows for better performance over more languages and code. 

- *Finetuning:* The base model is available for finetuning. This allows for more flexibility for use-cases where finetuning may be needed. 

- *Native Function Calling* - Like Mistral Large, this model has been trained on function calling. This makes it unique as being one of the first open source models to do so. 


### Kulinganisha Viganja Tokeni 

Katika sampuli hii, tutaangalia jinsi Mistral NeMo inavyoshughulikia ugawaji tokeni ukilinganisha na Mistral Large. 

Sampuli zote mbili zinachukua ombi lile lile lakini unapaswa kuona kuwa NeMo inarudisha tokeni chache zaidi ikilinganishwa na Mistral Large. 


In [11]:
pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 15.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
# Import needed packages:
from mistral_common.protocol.instruct.messages import (
    UserMessage,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import (
    Function,
    Tool,
)
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

# Load Mistral tokenizer

model_name = "open-mistral-nemo	"

tokenizer = MistralTokenizer.from_model(model_name)

# Tokenize a list of messages
tokenized = tokenizer.encode_chat_completion(
    ChatCompletionRequest(
        tools=[
            Tool(
                function=Function(
                    name="get_current_weather",
                    description="Get the current weather",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "description": "The temperature unit to use. Infer this from the users location.",
                            },
                        },
                        "required": ["location", "format"],
                    },
                )
            )
        ],
        messages=[
            UserMessage(content="What's the weather like today in Paris"),
        ],
        model=model_name,
    )
)
tokens, text = tokenized.tokens, tokenized.text

# Count the number of tokens
print(len(tokens))

128


In [13]:
# Import needed packages:
from mistral_common.protocol.instruct.messages import (
    UserMessage,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import (
    Function,
    Tool,
)
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

# Load Mistral tokenizer

model_name = "mistral-large-latest"

tokenizer = MistralTokenizer.from_model(model_name)

# Tokenize a list of messages
tokenized = tokenizer.encode_chat_completion(
    ChatCompletionRequest(
        tools=[
            Tool(
                function=Function(
                    name="get_current_weather",
                    description="Get the current weather",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "description": "The temperature unit to use. Infer this from the users location.",
                            },
                        },
                        "required": ["location", "format"],
                    },
                )
            )
        ],
        messages=[
            UserMessage(content="What's the weather like today in Paris"),
        ],
        model=model_name,
    )
)
tokens, text = tokenized.tokens, tokenized.text

# Count the number of tokens
print(len(tokens))

135


## Kujifunza hakukomi hapa, endelea Safari

Baada ya kumaliza somo hili, angalia mkusanyiko wetu wa [Kujifunza AI ya Kizazi](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) ili kuendelea kuongeza maarifa yako kuhusu AI ya Kizazi!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
